# Baseline Modeling and Preprocessing Across Healthcare Datasets

This notebook presents a baseline modeling and preprocessing workflow across three healthcare-related datasets: diabetes health indicators, life expectancy, and SUPPORT2 clinical data. The workflow includes dataset loading, basic inspection, missing-value checks, duplicate review, imputation, train-test splitting, baseline regression modeling, polynomial feature testing, and multicollinearity review using VIF.

The notebook is included in the modeling folder because its main purpose is model preparation and baseline model development rather than exploratory visualization. These baseline results helped guide later cleaning, EDA, and model-evaluation work in the project.


## Dataset Loading and Initial Inspection

This section loads the three datasets and displays their initial structure, dimensions, data types, and summary statistics before dataset-specific preprocessing and baseline modeling.


In [31]:
import pandas as pd

diabetes = pd.read_csv("diabetes_012_health_indicators_BRFSS2015.csv")
life_exp = pd.read_csv("Life Expectancy Data.csv")
support2 = pd.read_csv("support2.csv")

from IPython.display import display, Markdown

def show_head(name, df, n=5):
    display(Markdown(f"# {name}"))  # section title
    display(df.head(n))
    print()  # extra blank line

show_head("Diabetes (BRFSS 2015)", diabetes)
show_head("Life Expectancy", life_exp)
show_head("SUPPORT2", support2)

# Diabetes (BRFSS 2015)

,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


# Life Expectancy

,Country,Year,Status,Life expectancy,Adult Mortality,infant deaths,Alcohol,percentage expenditure,Hepatitis B,Measles,...,Polio,Total expenditure,Diphtheria,HIV/AIDS,GDP,Population,thinness 1-19 years,thinness 5-9 years,Income composition of resources,Schooling
0,Afghanistan,2015,Developing,65.0,263.0,62,0.01,71.279624,65.0,1154,...,6.0,8.16,65.0,0.1,584.259210,33736494.0,17.2,17.3,0.479,10.1
1,Afghanistan,2014,Developing,59.9,271.0,64,0.01,73.523582,62.0,492,...,58.0,8.18,62.0,0.1,612.696514,327582.0,17.5,17.5,0.476,10.0
2,Afghanistan,2013,Developing,59.9,268.0,66,0.01,73.219243,64.0,430,...,62.0,8.13,64.0,0.1,631.744976,31731688.0,17.7,17.7,0.470,9.9
3,Afghanistan,2012,Developing,59.5,272.0,69,0.01,78.184215,67.0,2787,...,67.0,8.52,67.0,0.1,669.959000,3696958.0,17.9,18.0,0.463,9.8
4,Afghanistan,2011,Developing,59.2,275.0,71,0.01,7.097109,68.0,3013,...,68.0,7.87,68.0,0.1,63.537231,2978599.0,18.2,18.2,0.454,9.5


# SUPPORT2

,age,sex,dzgroup,dzclass,num.co,edu,income,scoma,charges,totcst,...,ph,glucose,bun,urine,adlp,adls,adlsc,death,hospdead,sfdm2
0,62.84998,male,Lung Cancer,Cancer,0,11.0,$11-$25k,0.0,9715.0,NaN,...,7.459961,NaN,NaN,NaN,7.0,7.0,7.0,0,0,NaN
1,60.33899,female,Cirrhosis,COPD/CHF/Cirrhosis,2,12.0,$11-$25k,44.0,34496.0,NaN,...,7.250000,NaN,NaN,NaN,NaN,1.0,1.0,1,1,<2 mo. follow-up
2,52.74698,female,Cirrhosis,COPD/CHF/Cirrhosis,2,12.0,under $11k,0.0,41094.0,NaN,...,7.459961,NaN,NaN,NaN,1.0,0.0,0.0,1,0,<2 mo. follow-up
3,42.38498,female,Lung Cancer,Cancer,2,11.0,under $11k,0.0,3075.0,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,1,0,no(M2 and SIP pres)
4,79.88495,female,ARF/MOSF w/Sepsis,ARF/MOSF,1,NaN,NaN,26.0,50127.0,NaN,...,7.509766,NaN,NaN,NaN,NaN,2.0,2.0,0,0,no(M2 and SIP pres)


In [32]:

print(diabetes.shape, life_exp.shape, support2.shape)

(253680, 22) (2938, 22) (9105, 45)


In [33]:
diabetes.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 253680 entries, 0 to 253679
Data columns (total 22 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   Diabetes_012          253680 non-null  float64
 1   HighBP                253680 non-null  float64
 2   HighChol              253680 non-null  float64
 3   CholCheck             253680 non-null  float64
 4   BMI                   253680 non-null  float64
 5   Smoker                253680 non-null  float64
 6   Stroke                253680 non-null  float64
 7   HeartDiseaseorAttack  253680 non-null  float64
 8   PhysActivity          253680 non-null  float64
 9   Fruits                253680 non-null  float64
 10  Veggies               253680 non-null  float64
 11  HvyAlcoholConsump     253680 non-null  float64
 12  AnyHealthcare         253680 non-null  float64
 13  NoDocbcCost           253680 non-null  float64
 14  GenHlth               253680 non-null  float64
 15  

In [34]:
life_exp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2938 entries, 0 to 2937
Data columns (total 22 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Country                          2938 non-null   object 
 1   Year                             2938 non-null   int64  
 2   Status                           2938 non-null   object 
 3   Life expectancy                  2928 non-null   float64
 4   Adult Mortality                  2928 non-null   float64
 5   infant deaths                    2938 non-null   int64  
 6   Alcohol                          2744 non-null   float64
 7   percentage expenditure           2938 non-null   float64
 8   Hepatitis B                      2385 non-null   float64
 9   Measles                          2938 non-null   int64  
 10   BMI                             2904 non-null   float64
 11  under-five deaths                2938 non-null   int64  
 12  Polio               

In [35]:
support2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9105 entries, 0 to 9104
Data columns (total 45 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       9105 non-null   float64
 1   sex       9105 non-null   object 
 2   dzgroup   9105 non-null   object 
 3   dzclass   9105 non-null   object 
 4   num.co    9105 non-null   int64  
 5   edu       7471 non-null   float64
 6   income    6123 non-null   object 
 7   scoma     9104 non-null   float64
 8   charges   8933 non-null   float64
 9   totcst    8217 non-null   float64
 10  totmcst   5630 non-null   float64
 11  avtisst   9023 non-null   float64
 12  race      9063 non-null   object 
 13  sps       9104 non-null   float64
 14  aps       9104 non-null   float64
 15  surv2m    9104 non-null   float64
 16  surv6m    9104 non-null   float64
 17  hday      9105 non-null   int64  
 18  diabetes  9105 non-null   int64  
 19  dementia  9105 non-null   int64  
 20  ca        9105 non-null   obje

In [36]:
diabetes.describe() 

,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
count,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,...,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000
mean,0.296921,0.429001,0.424121,0.962670,28.382364,0.443169,0.040571,0.094186,0.756544,0.634256,...,0.951053,0.084177,2.511392,3.184772,4.242081,0.168224,0.440342,8.032119,5.050434,6.053875
std,0.698160,0.494934,0.494210,0.189571,6.608694,0.496761,0.197294,0.292087,0.429169,0.481639,...,0.215759,0.277654,1.068477,7.412847,8.717951,0.374066,0.496429,3.054220,0.985774,2.071148
min,0.000000,0.000000,0.000000,0.000000,12.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000
25%,0.000000,0.000000,0.000000,1.000000,24.000000,0.000000,0.000000,0.000000,1.000000,0.000000,...,1.000000,0.000000,2.000000,0.000000,0.000000,0.000000,0.000000,6.000000,4.000000,5.000000
50%,0.000000,0.000000,0.000000,1.000000,27.000000,0.000000,0.000000,0.000000,1.000000,1.000000,...,1.000000,0.000000,2.000000,0.000000,0.000000,0.000000,0.000000,8.000000,5.000000,7.000000
75%,0.000000,1.000000,1.000000,1.000000,31.000000,1.000000,0.000000,0.000000,1.000000,1.000000,...,1.000000,0.000000,3.000000,2.000000,3.000000,0.000000,1.000000,10.000000,6.000000,8.000000
max,2.000000,1.000000,1.000000,1.000000,98.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,5.000000,30.000000,30.000000,1.000000,1.000000,13.000000,6.000000,8.000000


In [37]:
life_exp.describe() 

,Year,Life expectancy,Adult Mortality,infant deaths,Alcohol,percentage expenditure,Hepatitis B,Measles,BMI,under-five deaths,Polio,Total expenditure,Diphtheria,HIV/AIDS,GDP,Population,thinness 1-19 years,thinness 5-9 years,Income composition of resources,Schooling
count,2938.000000,2928.000000,2928.000000,2938.000000,2744.000000,2938.000000,2385.000000,2938.000000,2904.000000,2938.000000,2919.000000,2712.00000,2919.000000,2938.000000,2490.000000,2.286000e+03,2904.000000,2904.000000,2771.000000,2775.000000
mean,2007.518720,69.224932,164.796448,30.303948,4.602861,738.251295,80.940461,2419.592240,38.321247,42.035739,82.550188,5.93819,82.324084,1.742103,7483.158469,1.275338e+07,4.839704,4.870317,0.627551,11.992793
std,4.613841,9.523867,124.292079,117.926501,4.052413,1987.914858,25.070016,11467.272489,20.044034,160.445548,23.428046,2.49832,23.716912,5.077785,14270.169342,6.101210e+07,4.420195,4.508882,0.210904,3.358920
min,2000.000000,36.300000,1.000000,0.000000,0.010000,0.000000,1.000000,0.000000,1.000000,0.000000,3.000000,0.37000,2.000000,0.100000,1.681350,3.400000e+01,0.100000,0.100000,0.000000,0.000000
25%,2004.000000,63.100000,74.000000,0.000000,0.877500,4.685343,77.000000,0.000000,19.300000,0.000000,78.000000,4.26000,78.000000,0.100000,463.935626,1.957932e+05,1.600000,1.500000,0.493000,10.100000
50%,2008.000000,72.100000,144.000000,3.000000,3.755000,64.912906,92.000000,17.000000,43.500000,4.000000,93.000000,5.75500,93.000000,0.100000,1766.947595,1.386542e+06,3.300000,3.300000,0.677000,12.300000
75%,2012.000000,75.700000,228.000000,22.000000,7.702500,441.534144,97.000000,360.250000,56.200000,28.000000,97.000000,7.49250,97.000000,0.800000,5910.806335,7.420359e+06,7.200000,7.200000,0.779000,14.300000
max,2015.000000,89.000000,723.000000,1800.000000,17.870000,19479.911610,99.000000,212183.000000,87.300000,2500.000000,99.000000,17.60000,99.000000,50.600000,119172.741800,1.293859e+09,27.700000,28.600000,0.948000,20.700000


In [38]:
support2.describe()

,age,num.co,edu,scoma,charges,totcst,totmcst,avtisst,sps,aps,...,sod,ph,glucose,bun,urine,adlp,adls,adlsc,death,hospdead
count,9105.000000,9105.000000,7471.000000,9104.000000,8.933000e+03,8217.000000,5630.000000,9023.000000,9104.000000,9104.000000,...,9104.000000,6821.000000,4605.000000,4753.000000,4243.000000,3464.000000,6238.000000,9105.000000,9105.000000,9105.000000
mean,62.650823,1.868644,11.747691,12.058546,5.999579e+04,30825.867768,28828.877838,22.610928,25.525872,37.597979,...,137.568541,7.415364,159.873398,32.349463,2191.546047,1.157910,1.637384,1.888272,0.681054,0.259198
std,15.593710,1.344409,3.447743,24.636694,1.026488e+05,45780.820986,43604.261932,13.233248,9.899377,19.903852,...,6.029326,0.080563,88.391541,26.792288,1455.245777,1.739672,2.231358,2.003763,0.466094,0.438219
min,18.041990,0.000000,0.000000,0.000000,1.169000e+03,0.000000,-102.719970,1.000000,0.199982,0.000000,...,110.000000,6.829102,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,52.797000,1.000000,10.000000,0.000000,9.740000e+03,5929.566400,5177.404300,12.000000,19.000000,23.000000,...,134.000000,7.379883,103.000000,14.000000,1165.500000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,64.856990,2.000000,12.000000,0.000000,2.502400e+04,14452.734400,13223.500000,19.500000,23.898438,34.000000,...,137.000000,7.419922,135.000000,23.000000,1968.000000,0.000000,1.000000,1.000000,1.000000,0.000000
75%,73.998960,3.000000,14.000000,9.000000,6.459800e+04,36087.937500,34223.601600,31.666656,30.199219,49.000000,...,141.000000,7.469727,188.000000,42.000000,3000.000000,2.000000,3.000000,3.000000,1.000000,1.000000
max,101.847960,9.000000,31.000000,100.000000,1.435423e+06,633212.000000,710682.000000,83.000000,99.187500,143.000000,...,181.000000,7.769531,1092.000000,300.000000,9000.000000,7.000000,7.000000,7.073242,1.000000,1.000000


## Diabetes Dataset Baseline Modeling


In [39]:
#check for empty cells
import pandas as pd

path = "diabetes_012_health_indicators_BRFSS2015.csv"

df = pd.read_csv(path)

total_empty_cells = int(df.isna().sum().sum())

print("Total empty cells:", total_empty_cells)


Total empty cells: 0


In [40]:
import pandas as pd

# file
path = "diabetes_012_health_indicators_BRFSS2015.csv"

# Load data
df = pd.read_csv(path)

# Count duplicates (beyond the first kept)
duplicate_rows = int(df.duplicated(keep="first").sum())
print("Duplicate rows to remove:", duplicate_rows)

# Drop duplicates
df_no_dup = df.drop_duplicates()

# Show before/after row counts
print("Rows before:", len(df))
print("Rows after :", len(df_no_dup))

#save cleaned file
df_no_dup.to_csv("diabetes_no_duplicates.csv", index=False)

Duplicate rows to remove: 23899
Rows before: 253680
Rows after : 229781


In [41]:
#single cell for everything
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

FILE = "diabetes_012_health_indicators_BRFSS2015.csv"
df = pd.read_csv(FILE).drop_duplicates()

TARGET = "Diabetes_012"
y = df[TARGET].astype(int)
X = df.drop(columns=[TARGET])

# Split 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

poly_degree = 2
interaction_only = False   
VIF_row_sample = 50000     
post_poly_max_feats = 200  
use_float32 = True         

def maybe_row_sample(df_like, max_rows):
    if len(df_like) > max_rows:
        return df_like.sample(n=max_rows, random_state=0)
    return df_like

def compute_vif(df_numeric: pd.DataFrame) -> pd.DataFrame:
    # VIF is column-focused; use a row subsample to speed it up
    df_small = maybe_row_sample(df_numeric, VIF_row_sample)
    Xc = sm.add_constant(df_small.values)
    vifs = [variance_inflation_factor(Xc, i) for i in range(1, Xc.shape[1])]
    return (pd.DataFrame({"feature": df_numeric.columns, "VIF": vifs})
            .sort_values("VIF", ascending=False)
            .reset_index(drop=True))

# Base-feature VIF on scaled features: train only
base_scaler = StandardScaler().fit(X_train)
X_train_base_scaled = pd.DataFrame(
    base_scaler.transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)
print("Top 10 base-feature VIFs (pre-polynomial):")
print(compute_vif(X_train_base_scaled).head(10))

# Pipeline: scale for polynomial+interactions for linear regression
poly = PolynomialFeatures(
    degree=poly_degree,
    include_bias=False,
    interaction_only=interaction_only
)
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("poly", poly),
    ("linreg", LinearRegression())
])

pipe.fit(X_train, y_train)

# Metrics
def metrics(y_true, y_pred, label):
    r2 = r2_score(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    print(f"{label} -> R^2: {r2:.4f} | RMSE: {rmse:.4f} | MAE: {mae:.4f}")

print("\nLinear Regression with degree-2 polynomials + interactions")
metrics(y_train, pipe.predict(X_train), "Train")
metrics(y_test,  pipe.predict(X_test),  "Test")

# Post-polynomial VIF 
# Transform train to poly space once using same scaler with poly
Z = pipe.named_steps["poly"].fit_transform(
        pipe.named_steps["scaler"].fit(X_train).transform(X_train)
    )
feat_names = pipe.named_steps["poly"].get_feature_names_out(X_train.columns)

#cast to float32 to reduce memory
if use_float32:
    Z = Z.astype(np.float32, copy=False)

# Limit number of features for VIF to keep it tractable
if Z.shape[1] > post_poly_max_feats:
    rng = np.random.RandomState(0)
    idx = rng.choice(Z.shape[1], size=post_poly_max_feats, replace=False)
    Z = Z[:, idx]
    feat_names = feat_names[idx]

# Build dataframe
Z_df = pd.DataFrame(Z, columns=feat_names, index=X_train.index)
Z_df = Z_df.loc[:, Z_df.std() > 1e-10]  # drop numerically-constant columns
Z_df_small = maybe_row_sample(Z_df, VIF_row_sample)

Zc = sm.add_constant(Z_df_small.values)
vifs_poly = [variance_inflation_factor(Zc, i) for i in range(1, Zc.shape[1])]
vif_poly_df = (pd.DataFrame({"feature": Z_df_small.columns, "VIF": vifs_poly})
               .sort_values("VIF", ascending=False)
               .reset_index(drop=True))

print("\nTop 10 VIFs after polynomial expansion (sampled):")
print(vif_poly_df.head(10))


Top 10 base-feature VIFs (pre-polynomial):
                feature       VIF
0               GenHlth  1.706886
1              PhysHlth  1.589630
2              DiffWalk  1.493955
3                Income  1.418837
4                   Age  1.354454
5                HighBP  1.306286
6             Education  1.268060
7              MentHlth  1.217384
8              HighChol  1.160673
9  HeartDiseaseorAttack  1.160016

Linear Regression with degree-2 polynomials + interactions
Train -> R^2: 0.1890 | RMSE: 0.6526 | MAE: 0.4451
Test -> R^2: 0.2004 | RMSE: 0.6479 | MAE: 0.4405



Top 10 VIFs after polynomial expansion (sampled):
                feature  VIF
0                 Sex^2  inf
1                Fruits  inf
2   HvyAlcoholConsump^2  inf
3                Stroke  inf
4             CholCheck  inf
5     HvyAlcoholConsump  inf
6           CholCheck^2  inf
7              Stroke^2  inf
8              Fruits^2  inf
9  HeartDiseaseorAttack  inf


### Diabetes Baseline Modeling Summary

The diabetes section reviews missing values and duplicate rows, then applies baseline linear regression modeling with polynomial feature testing and VIF review. This provides an initial benchmark for understanding model behavior and multicollinearity in the diabetes health indicators dataset.


## Life Expectancy Dataset Baseline Modeling


In [42]:
#check for empty cells
import pandas as pd

path = "Life Expectancy Data.csv"

df = pd.read_csv(path)

total_empty_cells = int(df.isna().sum().sum())

print("Total empty cells:", total_empty_cells)


Total empty cells: 2563


In [43]:
# mean_mode_impute
# Fill numeric columns with mean and categorical columns with mode.
# Saves an imputed copy of dataset.

import pandas as pd
import numpy as np
from pathlib import Path

#files
INPUT_CSV  = Path("Life Expectancy Data.csv")             
OUTPUT_CSV = Path("Life Expectancy Data - mean_mode_imputed.csv")

df = pd.read_csv(INPUT_CSV)

#Identify column types
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in df.columns if c not in num_cols]

#Fill numeric with column mean
if num_cols:
    means = df[num_cols].mean()
    df[num_cols] = df[num_cols].fillna(means)

#Fill categorical with column mode
def column_mode(s: pd.Series):
    m = s.mode(dropna=True)
    return m.iloc[0] if not m.empty else np.nan  

for c in cat_cols:
    if df[c].isna().any():
        mode_val = column_mode(df[c])
        if pd.isna(mode_val):
            continue 
        df[c] = df[c].fillna(mode_val)

#Quick report
report = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "nan_after": [int(df[c].isna().sum()) for c in df.columns],
})
print(report.to_string(index=False))

#Save
df.to_csv(OUTPUT_CSV, index=False)
# Saved output CSV using the relative filename above.

                         column   dtype  nan_after
                        Country  object          0
                           Year   int64          0
                         Status  object          0
               Life expectancy  float64          0
                Adult Mortality float64          0
                  infant deaths   int64          0
                        Alcohol float64          0
         percentage expenditure float64          0
                    Hepatitis B float64          0
                       Measles    int64          0
                           BMI  float64          0
             under-five deaths    int64          0
                          Polio float64          0
              Total expenditure float64          0
                    Diphtheria  float64          0
                       HIV/AIDS float64          0
                            GDP float64          0
                     Population float64          0
           thinness  1-19 years

In [44]:
import pandas as pd

# file
path = "Life Expectancy Data.csv"

# Load data
df = pd.read_csv(path)

# Count duplicates (beyond the first kept)
duplicate_rows = int(df.duplicated(keep="first").sum())
print("Duplicate rows to remove:", duplicate_rows)

# Drop duplicates
df_no_dup = df.drop_duplicates()

# Show before/after row counts
print("Rows before:", len(df))
print("Rows after :", len(df_no_dup))


Duplicate rows to remove: 0
Rows before: 2938
Rows after : 2938


In [45]:
# Life Expectancy — Linear Regression + Poly/Interactions + VIF
# Uses numeric + categorical features (categoricals via OrdinalEncoder; No one-hot)

import numpy as np, pandas as pd, re
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor

# setup
file_path = Path("Life Expectancy Data - mean_mode_imputed.csv")
rseed, test_size = 42, 0.2
use_poly, poly_degree, inter_only = True, 2, False   
VIF_max_rows, VIF_max_feats = 5000, 2500            

# load and normalize headers
df = pd.read_csv(file_path).drop_duplicates()
df["source_file"] = file_path.name

df.columns = (pd.Series(df.columns).astype(str)
              .str.replace("\xa0", " ", regex=False)
              .str.replace(r"\s+", " ", regex=True)
              .str.strip())

#resolve target issue
TARGET = next((c for c in df.columns if c.lower() == "life expectancy"),
              next((c for c in df.columns if "life" in c.lower() and "expect" in c.lower()), None))
assert TARGET is not None, f"No life expectancy column found. Got: {list(df.columns)}"

y = pd.to_numeric(df[TARGET], errors="coerce")
X = df.drop(columns=[TARGET])

#feature types
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

# if some "numeric-looking" strings are in categorical columns, coerce them
for c in cat_cols[:]:
    z = pd.to_numeric(X[c], errors="coerce")
    if z.notna().mean() > 0.95: 
        X[c] = z; num_cols.append(c); cat_cols.remove(c)

#drop rows with missing target
mask = y.notna()
X, y = X.loc[mask], y.loc[mask]

#split 
q = min(10, max(3, int(np.sqrt(len(y)) // 2)))
y_bins = pd.qcut(y, q=q, duplicates="drop")
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=test_size, random_state=rseed, stratify=y_bins)

# preprocessing
# Numerics: scale (poly/interaction)
num_steps = [("scale", StandardScaler())]
if use_poly and poly_degree > 1:
    num_steps.append(("poly", PolynomialFeatures(degree=poly_degree,
                                                 include_bias=False,
                                                 interaction_only=inter_only)))
num_pipe = Pipeline(num_steps)

# Categoricals: ordinal encode
cat_pipe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

pre = ColumnTransformer(
    [("num", num_pipe, num_cols)] +
    ([("cat", cat_pipe, cat_cols)] if cat_cols else []),
    remainder="drop"
)

#model
model = Pipeline([("pre", pre), ("est", LinearRegression())]).fit(X_tr, y_tr)

#metrics
def report(tag, yt, yp):
    rmse = np.sqrt(mean_squared_error(yt, yp))
    mae  = mean_absolute_error(yt, yp)
    r2   = r2_score(yt, yp)
    print(f"{tag:<5} | RMSE {rmse:.4f}  MAE {mae:.4f}  R² {r2:.4f}")

print("Target:", TARGET)
report("Train", y_tr, model.predict(X_tr))
report("Test",  y_te, model.predict(X_te))

#VIF helpers
def vif_table(Xm: np.ndarray, names: list[str], max_rows: int = 5000):
    if Xm.shape[0] > max_rows:
        rs = np.random.RandomState(RSEED)
        idx = rs.choice(Xm.shape[0], size=max_rows, replace=False)
        Xm = Xm[idx, :]
    rows = []
    for i in range(Xm.shape[1]):
        try:
            v = variance_inflation_factor(Xm, i)
        except Exception:
            v = np.nan
        rows.append((names[i], float(v) if np.isfinite(v) else np.nan))
    return pd.DataFrame(rows, columns=["feature","VIF"]).sort_values("VIF", ascending=False)

#Base VIFs (encoded cats + scaled numerics; no poly)
base_pre = ColumnTransformer(
    [("num", StandardScaler(), num_cols)] +
    ([("cat", cat_pipe, cat_cols)] if cat_cols else []),
    remainder="drop"
).fit(X_tr, y_tr)

Xb = base_pre.transform(X_tr)
base_names = list(num_cols) + list(cat_cols)
print("\nTop VIFs — base (numerics + encoded categoricals):")
print(vif_table(np.asarray(Xb, float), base_names, VIF_max_rows).head(20).to_string(index=False))

# Post-poly VIFs (numerics-only poly + encoded cats)
if use_poly and poly_degree > 1:
    # Rebuild numeric transform from the trained pipeline
    num_tr = model.named_steps["pre"].named_transformers_["num"]
    Xn = num_tr.transform(X_tr[num_cols]) if num_cols else np.empty((len(X_tr), 0))
    # encoded cats
    if cat_cols:
        cat_tr = model.named_steps["pre"].named_transformers_["cat"]
        Xc = cat_tr.transform(X_tr[cat_cols])
        names_c = list(cat_cols)
    else:
        Xc = np.empty((len(X_tr), 0))
        names_c = []

    #feature names for numeric poly
    if "poly" in num_tr.named_steps:
        poly = num_tr.named_steps["poly"]
        names_n = list(poly.get_feature_names_out(num_cols))
    else:
        names_n = list(num_cols)

    Xa = np.hstack([Xn, Xc]) if Xc.size else Xn
    names_all = names_n + names_c

    if Xa.shape[1] <= VIF_max_feats:
        print("\nTop VIFs — after polynomial/interactions:")
        print(vif_table(np.asarray(Xa, float), names_all, VIF_max_rows).head(20).to_string(index=False))
    else:
        print(f"\n(Skipped post-poly VIF: {Xa.shape[1]} columns > {VIF_max_feats})")

#quick look
if cat_cols:
    print("\nCategoricals included via OrdinalEncoder:", cat_cols)
else:
    print("\nNo categorical features detected.")
print("Numeric features used:", num_cols)

Target: Life expectancy
Train | RMSE 2.4954  MAE 1.8250  R² 0.9306
Test  | RMSE 3.0012  MAE 2.1235  R² 0.9029

Top VIFs — base (numerics + encoded categoricals):
                        feature        VIF
                  infant deaths 198.282153
              under-five deaths 195.597364
             thinness 5-9 years   9.282096
            thinness 1-19 years   9.050834
                            GDP   5.173000
         percentage expenditure   5.012562
                      Schooling   3.398777
Income composition of resources   3.094258
                         Status   3.083669
                        Country   2.861436
                     Diphtheria   2.233189
                          Polio   1.973300
                            BMI   1.719549
                Adult Mortality   1.694312
                        Alcohol   1.685306
                     Population   1.521945
                    Hepatitis B   1.412935
                       HIV/AIDS   1.401224
                     

                                          feature          VIF
                  infant deaths under-five deaths 6.906009e+06
                                  infant deaths^2 1.782782e+06
                              under-five deaths^2 1.688420e+06
                 infant deaths thinness 5-9 years 4.378127e+04
                infant deaths thinness 1-19 years 4.300249e+04
                         infant deaths Population 3.664690e+04
         percentage expenditure under-five deaths 3.639809e+04
             under-five deaths thinness 5-9 years 3.612745e+04
            under-five deaths thinness 1-19 years 3.527530e+04
             infant deaths percentage expenditure 3.524387e+04
                     under-five deaths Population 3.322880e+04
                                under-five deaths 2.340160e+04
                                    infant deaths 2.171597e+04
                            under-five deaths GDP 1.730600e+04
                                infant deaths GDP 1.716

### Life Expectancy Baseline Modeling Summary

The life expectancy section uses mean/mode imputation, duplicate review, train-test splitting, and baseline regression modeling. Polynomial features and VIF diagnostics are used to evaluate model performance and identify multicollinearity among predictors.


## SUPPORT2 Dataset Baseline Modeling


In [46]:
#check for empty cells
import pandas as pd

path = "support2.csv"

df = pd.read_csv(path)

total_empty_cells = int(df.isna().sum().sum())

print("Total empty cells:", total_empty_cells)

Total empty cells: 47110


In [47]:
import pandas as pd

# file
path = "support2.csv"

# Load data
df = pd.read_csv(path)

# Count duplicates (beyond the first kept)
duplicate_rows = int(df.duplicated(keep="first").sum())
print("Duplicate rows to remove:", duplicate_rows)

# Drop duplicates
df_no_dup = df.drop_duplicates()

# Show before/after row counts
print("Rows before:", len(df))
print("Rows after :", len(df_no_dup))


Duplicate rows to remove: 0
Rows before: 9105
Rows after : 9105


In [48]:
# impute_mean_mode_support2
import pandas as pd
import numpy as np

# Load 
df = pd.read_csv("support2.csv")

#Treat common NA-like & empty strings as missing
NA_like = {"na", "nan", "n/a", "none", "null", "missing", "?"}
for c in df.columns:
    if df[c].dtype == object or pd.api.types.is_string_dtype(df[c]):
        s = df[c].astype(str).str.strip()
        df[c] = s.mask((s == "") | (s.str.lower().isin(NA_like)))  

#Impute: mean for numeric, mode for categorical
num_cols = df.select_dtypes(include="number").columns
cat_cols = df.columns.difference(num_cols)

# numeric: mean
for c in num_cols:
    df[c] = df[c].fillna(df[c].mean())

# categorical: mode
for c in cat_cols:
    m = df[c].mode(dropna=True)
    fill_val = m.iloc[0] if not m.empty else "Unknown"
    df[c] = df[c].fillna(fill_val)

#Save
df.to_csv("support2_imputed.csv", index=False)
print("Saved: support2_imputed.csv")

Saved: support2_imputed.csv


In [49]:
# SUPPORT2 — Linear Regression + Poly/Interactions + Base VIF (sampled, No one-hot)

import numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.feature_selection import VarianceThreshold
from statsmodels.stats.outliers_influence import variance_inflation_factor as VIF

# setup
file = Path("support2_imputed.csv")
target = None             
rseed, test_size = 42, 0.2
degree = 2               
VIF_max_rows = 3000       
max_cat_levels = 200     

df = pd.read_csv(file).drop_duplicates()
df.columns = (pd.Series(df.columns).astype(str)
              .str.replace("\xa0"," ", regex=False).str.replace(r"\s+"," ", regex=True).str.strip())
df["source_file"] = file.name  

# Resolve target issues
if target is None:
    cands = ["death","mortality","survtime","surv_days","survday","length_of_stay","los","outcome","score","target","y"]
    target = next((c for c in df.columns if c.lower() in cands), None)
    if target is None:
        nv = df.select_dtypes(include=[np.number]).var().sort_values(ascending=False)
        target = nv.index[0] if len(nv) else None
assert target in df.columns, f"Set TARGET manually. Columns: {list(df.columns)}"
print(f"Target: {target} | File: {file.name}")

# y, X, feature 
y = pd.to_numeric(df[target], errors="coerce")
X = df.drop(columns=[target])

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

#numeric-looking categoricals to numeric
for c in cat_cols[:]:
    z = pd.to_numeric(X[c], errors="coerce")
    if z.notna().mean() > 0.95:
        X[c] = z; num_cols.append(c); cat_cols.remove(c)

# Drop huge-cardinality categoricals: ID-like that is unique
cat_cols = [c for c in cat_cols if X[c].nunique() <= max_cat_levels]

# Keep rows with target present
mask = y.notna()
X, y = X.loc[mask], y.loc[mask]

# Remove zero-variance numerics for stability and speed
if num_cols:
    vt = VarianceThreshold(0.0)
    keep = vt.fit(X[num_cols]).get_support()
    num_cols = [c for c, k in zip(num_cols, keep) if k]

# Split
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=test_size, random_state=rseed)

# Preprocessing: numerics- scale + poly; categoricals -> OrdinalEncoder (No one-hot)
num_pipe = Pipeline([
    ("scale", StandardScaler()),
    ("poly",  PolynomialFeatures(degree=degree, include_bias=False, interaction_only=False))
])
cat_pipe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

pre = ColumnTransformer(
    [("num", num_pipe, num_cols)] + ([("cat", cat_pipe, cat_cols)] if cat_cols else []),
    remainder="drop"
)

# Model
model = Pipeline([("pre", pre), ("est", LinearRegression())]).fit(Xtr, ytr)

# Metrics
def report(tag, yt, yp):
    rmse = np.sqrt(mean_squared_error(yt, yp))
    print(f"{tag:<5} | RMSE {rmse:.4f}  MAE {mean_absolute_error(yt, yp):.4f}  R² {r2_score(yt, yp):.4f}")

print("\n=== Performance ===")
report("Train", ytr, model.predict(Xtr))
report("Test",  yte, model.predict(Xte))

# Multicollinearity
# Base design 
base_pre = ColumnTransformer(
    [("num", StandardScaler(), num_cols)] + ([("cat", cat_pipe, cat_cols)] if cat_cols else []),
    remainder="drop"
).fit(Xtr, ytr)

Xb = base_pre.transform(Xtr)
# Row sampling for speed
if Xb.shape[0] > VIF_max_rows:
    rng = np.random.RandomState(rseed)
    Xb = Xb[rng.choice(Xb.shape[0], VIF_max_rows, replace=False)]

base_names = list(num_cols) + list(cat_cols)

def vif_table(M: np.ndarray, names: list[str]) -> pd.DataFrame:
    M = np.asarray(M, float)
    rows = []
    for i, name in enumerate(names):
        try:
            v = float(VIF(M, i))
        except Exception:
            v = np.nan
        rows.append((name, v))
    return (pd.DataFrame(rows, columns=["feature","VIF"])
              .replace([np.inf,-np.inf], np.nan)
              .sort_values("VIF", ascending=False))

vif_base = vif_table(Xb, base_names)
print("\nTop VIFs — base design (sampled rows; first 20)")
print(vif_base.head(20).to_string(index=False))

# Info
print("\nCategoricals (ordinal-encoded):", cat_cols if cat_cols else "(none)")
print("Numeric features:", num_cols)

Target: death | File: support2_imputed.csv

=== Performance ===
Train | RMSE 0.3469  MAE 0.2732  R² 0.4443
Test  | RMSE 0.3848  MAE 0.3052  R² 0.3264

Top VIFs — base design (sampled rows; first 20)
 feature       VIF
  surv2m 36.065142
  surv6m 24.864961
   adlsc  8.277089
    adls  7.256885
    race  7.125160
     sps  6.495438
   prg2m  6.454525
   prg6m  6.212430
      ca  4.918852
     dnr  4.714990
 dzgroup  4.568434
 dzclass  4.096455
  income  4.004179
  totcst  3.974234
     aps  3.720849
   sfdm2  3.043498
   scoma  2.860963
 charges  2.811618
 avtisst  2.642301
hospdead  2.405434

Categoricals (ordinal-encoded): ['sex', 'dzgroup', 'dzclass', 'income', 'race', 'ca', 'dnr', 'sfdm2', 'source_file']
Numeric features: ['age', 'num.co', 'edu', 'scoma', 'charges', 'totcst', 'totmcst', 'avtisst', 'sps', 'aps', 'surv2m', 'surv6m', 'hday', 'diabetes', 'dementia', 'prg2m', 'prg6m', 'dnrday', 'meanbp', 'wblc', 'hrt', 'resp', 'temp', 'pafi', 'alb', 'bili', 'crea', 'sod', 'ph', 'glucose',

### SUPPORT2 Baseline Modeling Summary

The SUPPORT2 section handles missing clinical data with mean/mode imputation, reviews duplicates, and builds a baseline regression model using encoded categorical and numeric predictors. VIF diagnostics are included to identify features with potential multicollinearity.
